In [2]:
# ============================================================
# BattingEdge V9.5 - XGBOOST Model Training
# Features: 99 raw pose + 8 angles (NO velocities)
# Target: 83-86% accuracy
# ============================================================

import numpy as np
import pickle
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib

# XGBoost Imports
from xgboost import XGBClassifier

# ================= CONFIG =================
FEATURE_DIR = Path(r"D:\Users\Anoshia\BattingEdge_FYP\features")
MODEL_DIR   = Path(r"D:\Users\Anoshia\BattingEdge_FYP\backend\models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Algorithm Name for File Saving
ALGO_NAME = "xgboost"

# XGBoost Hyperparameters
EPOCHS = 100  # Equivalent to n_estimators
BATCH_SIZE = 16 # Not applicable to XGBoost standard training, but kept for config consistency
LEARNING_RATE = 0.1
# ==========================================

print("="*70)
print(f"BATTINGEDGE V9.5 - MODEL TRAINING ({ALGO_NAME.upper()})")
print("="*70)
print()

# ================= LOAD DATA =================
print("📂 Loading data...")

X_train = np.load(FEATURE_DIR / "X_train.npy")
y_train = np.load(FEATURE_DIR / "y_train.npy")
X_val   = np.load(FEATURE_DIR / "X_val.npy")
y_val   = np.load(FEATURE_DIR / "y_val.npy")
X_test  = np.load(FEATURE_DIR / "X_test.npy")
y_test  = np.load(FEATURE_DIR / "y_test.npy")

with open(FEATURE_DIR / "classes.pkl", "rb") as f:
    CLASSES = pickle.load(f)

num_classes = len(CLASSES)
N, T, F = X_train.shape

print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Classes: {CLASSES}")
print(f"Features per frame: {F} (99 pose + 8 angles)")
print()

# ================= CRITICAL: SCALING =================
print("⚖️  Applying StandardScaler...")

scaler = StandardScaler()

# Fit on training data (flatten to 2D)
N_train = X_train.shape[0]
X_train_2d = X_train.reshape(N_train * T, F)
scaler.fit(X_train_2d)

def scale_data(X):
    N, T, F = X.shape
    X_2d = X.reshape(N * T, F)
    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(N, T, F)

X_train = scale_data(X_train)
X_val   = scale_data(X_val)
X_test  = scale_data(X_test)

# Save scaler and classes with algorithm name
scaler_path = MODEL_DIR / f"scaler_V9_5_{ALGO_NAME}.pkl"
classes_path = MODEL_DIR / f"classes_V9_5_{ALGO_NAME}.pkl"

joblib.dump(scaler, scaler_path)
joblib.dump(CLASSES, classes_path)
print(f"✅ Scaler saved: {scaler_path.name}")
print(f"✅ Classes saved: {classes_path.name}")
print()

# ================= FLATTENING (REQUIRED FOR XGBOOST) =================
print("Flattening 3D sequential data for XGBoost (N, T*F)...")
# Reshape from (N, 50, 107) -> (N, 5350)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)
print(f"New Input Shape: {X_train_flat.shape}")
print()

# ================= CLASS WEIGHTS =================
print("⚖️  Computing class weights...")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

# Apply weights to samples for XGBoost
sample_weights = np.ones(y_train.shape[0])
for i in range(y_train.shape[0]):
    sample_weights[i] = class_weight_dict[y_train[i]]

for i, cls in enumerate(CLASSES):
    print(f"  {cls:15s}: {class_weight_dict[i]:.3f}")
print()

# ================= MODEL =================
print("🏗️  Building XGBoost model...")

model = XGBClassifier(
    n_estimators=EPOCHS,
    learning_rate=LEARNING_RATE,
    max_depth=6,              # Standard depth
    objective='multi:softprob',
    num_class=num_classes,
    tree_method='hist',       # Faster training
    eval_metric=['merror', 'mlogloss'],
    early_stopping_rounds=10
)

print(model)
print()

# ================= TRAINING =================
print("="*70)
print("🚀 STARTING TRAINING")
print("="*70)
print()

# XGBoost fit
model.fit(
    X_train_flat, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_train_flat, y_train), (X_val_flat, y_val)],
    verbose=True
)

results = model.evals_result()

print()
print("="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print()

# Save final model (using joblib for XGBoost)
final_model_path = MODEL_DIR / f"battingedge_V9_5_{ALGO_NAME}_best.json" # XGBoost native format
model.save_model(final_model_path)
print(f"💾 Saved final model: {final_model_path.name}")
print()

# ================= EVALUATION =================
print("="*70)
print("📊 EVALUATING ON TEST SET")
print("="*70)
print()

# Predict
y_pred = model.predict(X_test_flat)

# Classification report
print("📋 CLASSIFICATION REPORT:")
print()
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(report)

report_path = MODEL_DIR / f"report_V9_5_{ALGO_NAME}.txt"
with open(report_path, "w") as f:
    f.write(report)
print(f"✅ Report saved: {report_path.name}")
print()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("🔢 CONFUSION MATRIX:")
print("   (Rows = True, Cols = Predicted)")
print()
print("        ", "  ".join([f"{cls[:4]:>4s}" for cls in CLASSES]))
for i, cls in enumerate(CLASSES):
    print(f"{cls[:8]:8s}", "  ".join([f"{cm[i,j]:4d}" for j in range(num_classes)]))
print()

# Per-class accuracy
print("📈 PER-CLASS ACCURACY:")
print()
for i, cls in enumerate(CLASSES):
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"   {cls:15s}: {correct:3d}/{total:3d} = {accuracy:5.1f}%")

overall_acc = np.trace(cm) / np.sum(cm) * 100
print()
print(f"   {'OVERALL':15s}: {np.trace(cm):3d}/{np.sum(cm):3d} = {overall_acc:5.2f}%")
print()

# Major confusions
print("🔍 MAJOR CONFUSIONS (>3 cases):")
print()
confusions = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 3:
            confusions.append((CLASSES[i], CLASSES[j], cm[i, j]))

if confusions:
    confusions.sort(key=lambda x: x[2], reverse=True)
    for true_cls, pred_cls, count in confusions:
        print(f"   {true_cls:15s} → {pred_cls:15s}: {count} cases")
else:
    print("   ✅ No major confusions!")
print()

# ================= VISUALIZATIONS =================
print("📊 Generating visualizations...")

# Confusion matrix heatmap
cm_path = MODEL_DIR / f"confusion_matrix_V9_5_{ALGO_NAME}.png"
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Confusion Matrix - V9.5 ({ALGO_NAME})", fontsize=14, fontweight='bold')
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(cm_path, dpi=300)
print(f"   ✅ Saved: {cm_path.name}")
plt.close()

# Training history
history_path = MODEL_DIR / f"training_history_V9_5_{ALGO_NAME}.png"
epochs_len = len(results['validation_0']['mlogloss'])
epochs_range = range(epochs_len)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
# XGBoost calculates error (merror), so accuracy = 1 - error
train_acc = [1 - x for x in results['validation_0']['merror']]
val_acc = [1 - x for x in results['validation_1']['merror']]
plt.plot(epochs_range, train_acc, label='Train')
plt.plot(epochs_range, val_acc, label='Val')
plt.xlabel('Iterations')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, results['validation_0']['mlogloss'], label='Train')
plt.plot(epochs_range, results['validation_1']['mlogloss'], label='Val')
plt.xlabel('Iterations')
plt.ylabel('Log Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(history_path, dpi=300)
print(f"   ✅ Saved: {history_path.name}")
plt.close()

print()

# ================= COMPARISON =================
print("="*70)
print(f"🎉 V9.5 ({ALGO_NAME}) TRAINING COMPLETE")
print("="*70)
print()

# Save metadata
metadata = {
    "version": "V9.5",
    "algorithm": ALGO_NAME,
    "features": "99 raw pose + 8 angles (no velocities) - XGBoost Flattened",
    "classes": CLASSES,
    "test_accuracy": float(overall_acc),
    "per_class_accuracy": {
        CLASSES[i]: float((cm[i,i] / cm[i,:].sum() * 100) if cm[i,:].sum() > 0 else 0)
        for i in range(num_classes)
    },
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE
    }
}

metadata_path = MODEL_DIR / f"metadata_V9_5_{ALGO_NAME}.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("📦 Saved artifacts:")
print(f"   1. Final model: {final_model_path.name}")
print(f"   2. Scaler: {scaler_path.name}")
print(f"   3. Classes: {classes_path.name}")
print(f"   4. Report: {report_path.name}")
print(f"   5. Confusion matrix: {cm_path.name}")
print(f"   6. Training history: {history_path.name}")
print(f"   7. Metadata: {metadata_path.name}")
print()

if overall_acc >= 83:
    print("✨ EXCELLENT RESULT! Ready for deployment.")
elif overall_acc >= 81:
    print("✅ GOOD RESULT! High accuracy.")
elif overall_acc >= 79:
    print("→ ACCEPTABLE. Stable performance.")
else:
    print("⚠️ Below target. Review feature extraction.")

print("="*70)

BATTINGEDGE V9.5 - MODEL TRAINING (XGBOOST)

📂 Loading data...
Train: 3007 samples
Val:   388 samples
Test:  378 samples
Classes: ['Cover Drive', 'Cut Shot', 'Defense', 'Pull Shot', 'Sweep Shot']
Features per frame: 107 (99 pose + 8 angles)

⚖️  Applying StandardScaler...
✅ Scaler saved: scaler_V9_5_xgboost.pkl
✅ Classes saved: classes_V9_5_xgboost.pkl

Flattening 3D sequential data for XGBoost (N, T*F)...
New Input Shape: (3007, 5350)

⚖️  Computing class weights...
  Cover Drive    : 1.004
  Cut Shot       : 1.018
  Defense        : 0.972
  Pull Shot      : 0.999
  Sweep Shot     : 1.009

🏗️  Building XGBoost model...
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=10,
              enable_categorical=False, eval_metric=['merror', 'mlogloss'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None